In [ ]:
# # check the machine

# !nproc
# !free -g | head -2
# !pip list 2>/dev/null | grep -i -E "^(torch|transformers|safetensors|opencv)"
# !ls /kaggle/input/datasets/saisdsaiiitbh/

In [ ]:
# !cp /kaggle/input/datasets/saisdsaiiitbh/dff-code/*.py /kaggle/working/
# %cd /kaggle/working
# !ls

In [ ]:
# !python test_dff_adapter.py --real

In [ ]:
# # a short benchmark run on real data. This runs one epoch with 1 random frame per video (3,600 samples):

# !python train.py --backbone small \
#   --crops_root /kaggle/input/datasets/saisdsaiiitbh/ff-crops \
#   --out_dir runs/bench --epochs 1 --frames_per_video 1 \
#   --batch_size 32 --num_workers 4 --val_frames 2 --log_every 10

In [ ]:
# # compute the speed.

# import json
# rows = [json.loads(l) for l in open("runs/bench/log.jsonl")]
# tr = [r for r in rows if "loss_auth" in r]
# print("steps:", tr[-1]["step"], "minutes:", tr[-1]["time_min"])
# print("samples/sec (both passes):", tr[-1]["step"]*32 / (tr[-1]["time_min"]*60))

In [ ]:
# import json
# rows = [json.loads(l) for l in open("runs/bench/log.jsonl")]
# tr = [r for r in rows if "loss_auth" in r]
# a, b = tr[0], tr[-1]
# print("first logged row at", a["time_min"], "min (everything before = listing + model load)")
# print("steady-state samples/s:", (b["step"]-a["step"])*32/((b["time_min"]-a["time_min"])*60))

In [ ]:
# # gpu speed alone

# import time, torch, torch.nn.functional as F
# from train import build_model
# m = build_model("small").cuda().train()
# opt = torch.optim.Adam(m.trainable_parameters(), 2e-4)
# scaler = torch.amp.GradScaler("cuda")
# x = torch.randn(32,3,224,224,device="cuda")
# ya = torch.randint(0,2,(32,),device="cuda").float()
# yt = torch.randint(0,5,(32,),device="cuda")
# def step():
#     with torch.autocast("cuda", dtype=torch.float16):
#         l = 10*F.binary_cross_entropy_with_logits(m(x,0).float(), ya) + 2*F.cross_entropy(m(x,1).float(), yt)
#     opt.zero_grad(); scaler.scale(l).backward(); scaler.step(opt); scaler.update()
# for _ in range(3): step()
# torch.cuda.synchronize(); t = time.time()
# for _ in range(20): step()
# torch.cuda.synchronize()
# print("GPU-only samples/s:", 20*32/(time.time()-t))

In [ ]:
# # data loading alone, no model

# import time
# from torch.utils.data import DataLoader
# from dataset import FrameDataset
# ROOT = "/kaggle/input/datasets/saisdsaiiitbh/ff-crops"
# t = time.time()
# ds = FrameDataset(ROOT, ROOT+"/ffpp_manifest_split.csv", "train", "train", 8)
# print("dataset init (listing folders): %.1fs" % (time.time()-t))
# for nw in (4, 8, 12):
#     it = iter(DataLoader(ds, batch_size=32, shuffle=True, num_workers=nw))
#     for _ in range(3): next(it)            # warm-up
#     t = time.time()
#     for _ in range(30): next(it)
#     print(f"{nw} workers: {30*32/(time.time()-t):.0f} samples/s (data only)")
#     del it

In [ ]:
!cp /kaggle/input/datasets/saisdsaiiitbh/dff-code/*.py /kaggle/working/
%cd /kaggle/working

In [ ]:
!python -u run_experiments.py dff_full dff_noaux